---
title: Homework 3 - Example Answers
subtitle: Applied Pandas - Group Operations
date: last-modified
from: markdown+emoji
---

In [2]:
import pandas as pd
# from google.colab import data_table
# data_table.enable_dataframe_formatter()

# !pip install itables
from itables import init_notebook_mode
from itables import show

# Direction

- Please use the Homework 3 Jupyter Notebook, `danl-210-hw-03-q.ipynb`, available from Brightspace.

- Please submit your **Jupyter Notebook for Homework 3** to the Brightspace with the name below:

  - `danl-210-hw3-LASTNAME-FIRSTNAME.ipynb`\
  ( e.g., `danl-210-hw3-choe-byeonghak.ipynb` )

- The due is March 27, 2024, 11:59 P.M.

- Please send an email to Byeong-Hak (`bchoe@geneseo.edu`) if you have any questions or clarifications.

<br><br><br>

# Part 1 - Magnificent Seven



<br>

<p align="center">
  <img src="https://bcdanl.github.io/lec_figs/stocks.png" width="600px">
</p>

<br>

Consider the `stock` DataFrame for Part 1.

In [3]:
path = 'https://bcdanl.github.io/data/stock_2019_2024.csv'
stock = pd.read_csv(path)

show(stock)

In [8]:
stock_company = stock.groupby('company')

## Question 1

What are the minimum, first quartile, median, thrid quartile, maximum, mean, and standard deviation of `Close` and `Volume` for each `company`?

*Answer*:

In [9]:
q1_close = (
    stock_company[['Close']]
    .describe()
)

show(q1_close)

In [10]:
q1_volume = (
    stock_company[['Volume']]
    .describe()
)

show(q1_volume)

<br>

## Question 2

Find the 10 largest values for `Volume` for each `company` using a lambda function.

*Answer*:

In [11]:
q2 = (
    stock_company
    .apply(lambda df: df.nlargest(10, 'Volume', keep = "all"))
)

show(q2)

<br>

## Question 3

- Calculate the Z-score of `Close` for each `company`  on each `Date` $t$.
  - The formula for the Z-score of `Close` for each `company` on `Date` $t$ is as follows:
  
$$
z_{t} = \frac{\text{Close}_{t} - \text{mean}(\text{Close}_{t})}{\text{std}(\text{Close}_{t})},
$$

where

  - $\text{Close}_{t}$ is a company's `Close` price on date `t`;
  - $\text{mean}(\text{Close}_{t})$ is the mean value of `Close` for a company;
  - $\text{std}(\text{Close}_{t})$ is the standard deviation of `Close` for a company.


*Answer*:

In [15]:
q3 = (
    stock
    .assign(
        Close_z = ( stock['Close'] - stock_company['Close'].transform('mean') )
                  / stock_company['Close'].transform('std'),
        Close_z_all =
         (stock['Close'] - stock['Close'].mean())
          / stock['Close'].std()  # for comparison
    )
    [['company', 'Date', 'Close', 'Close_z', 'Close_z_all']]
)

show(q3)

<br>

## Question 4

- Use the `transform()` method on the `stock` data to represent all the values of `Open`, `High`, `Low`, `Close`, `Adj Close`, and `Volume` in terms of the first date in the data.

- To do so, divide all values for each company by the values of the first date in the data for that company.

*Answer*:

In [16]:
q4 = (
    stock
    .assign(
        open_rel = stock['Open'] / stock_company['Open'].transform('first'),
        high_rel = stock['High'] / stock_company['High'].transform('first'),
        low_rel = stock['Low'] / stock_company['Low'].transform('first'),
        close_rel = stock['Close'] / stock_company['Close'].transform('first'),
        adj_close_rel = stock['Adj Close'] / stock_company['Adj Close'].transform('first'),
        volume_rel = stock['Volume'] / stock_company['Volume'].transform('first'),
    )[['Date', 'company', 'open_rel', 'high_rel', 'low_rel',
       'close_rel', 'adj_close_rel', 'volume_rel']]
)

show(q4)

<br><br><br>


## Question 5

- Calculate the difference between the highest and lowest `Close` for each company.
  - Which company experienced the largest range in its stock price, and what was that range?

*Answer*:

In [18]:
q5 = (
    stock_company
    .agg(
        Close_max = ('Close', 'max'),
        Close_min = ('Close', 'min')
    )
    .assign(
        Close_range = lambda x: x['Close_max'] - x['Close_min']
    )
)

show(q5)

In [20]:
q5_company = q5.nlargest(1, "Close_range", keep = "all")

show(q5_company)

<br><br><br>





# Part 2 - Search Engine Marketing


<br>

<p align="center">
  <img src="https://bcdanl.github.io/lec_figs/ebay-google-mkt.png" width="800px">
</p>

<br>


Load DataFrame for Part 2:

In [21]:
paidsearch = pd.read_csv('https://bcdanl.github.io/data/paidsearch.csv')

show(paidsearch)

##  Variable Description

- `dma`: an identification number of a designated market (DM) area `i` (e.g., Boston, Los Angeles)
- `treatment_period`: 0 if date is before May 22, 2012 and 1 after.
- `search_stays_on`: 1 if the paid-search goes off in dma `i`, 0 otherwise.
- `revenue`: eBay's sales revenue for dma `i` and date `t`

<br>

## Question 6

Summarize the mean value of `revenue` for each group of `search_stays_on` and for each `date`.

*Answer*:

In [24]:
q6 = (
    paidsearch
    .groupby(['date', 'search_stays_on'])
    .agg(revenue_mean = ('revenue', 'mean'))
    .reset_index()
)

show(q6)

<br>

## Question 7

Calculate the log difference between mean revenues in each group of `search_stays_on`. (This is the log of the average revenue in group of `search_stays_on` == 1 minus the log of the average revenue in group of `search_stays_on` == 0.)

- For example, consider the following two observations:

In [ ]:
# date        the daily mean vale of `revenue`   search_stays_on
# 1-Apr-12    93650.68                           0
# 1-Apr-12    120277.57                          1

- The log difference of daily mean revenues between the two group of `search_stays_on` for date 1-Apr-12 is log(120277.57) - log(93650.68).

- *Hint*: Below example shows how we can calculate the log of `Value` using `np.log()`:

In [ ]:
import pandas as pd
import numpy as np

# Creating a DataFrame
data = {'Value': [1, 10, 100, 1000, 10000]}
df = pd.DataFrame(data)

# Calculating logarithms
df['Log_natural'] = np.log(df['Value'])

*Answer*:

In [28]:
# reset_index() is necessary
q7_search_stays_on = q6.query('search_stays_on == 1').reset_index(drop = True)
q7_search_stays_off = q6.query('search_stays_on == 0').reset_index(drop = True)

In [29]:
show(q7_search_stays_on)

In [30]:
show(q7_search_stays_off)

In [36]:
import numpy as np

q7 = (
    q7_search_stays_on
    .assign(
        revenue_mean_diff = np.log(q7_search_stays_on['revenue_mean']) - np.log(q7_search_stays_off['revenue_mean'])
    )[['date', 'revenue_mean_diff']]
)

show(q7)

<br><br>



# Part 3 - NFL

<br>

<p align="center">
  <img src="https://bcdanl.github.io/lec_figs/nfl.png" width="300px">
</p>

<br>

- The following is the DataFrame for Part 3.

In [38]:
NFL2023_stuffs = pd.read_csv('https://bcdanl.github.io/data/NFL2023_stuffs.csv')

show(NFL2023_stuffs)


- `NFL2023_stuffs` is the DataFrame that contains information about NFL games in year 2022, in which the unit of observation is a single play for each drive in a NFL game.


## Variable description

- `play_id`: Numeric play identifier that when used with `game_id` and `drive` provides the unique identifier for a single play
- `game_id`: Ten digit identifier for NFL game.
- `drive`: Numeric drive number in the game.
- `week`: Season week.
- `posteam`: String abbreviation for the team with possession.
- `qtr`: Quarter of the game (5 is overtime).
- `half_seconds_remaining`: Numeric seconds remaining in the half.
- `down`: The down for the given play.
  - Basically you get four attempts (aka downs) to move the ball 10 yards (by either running with it or passing it).
  - If you make 10 yards then you get another set of four downs.
- `pass`: Binary indicator if the play was a pass play.
- `wp`: Estimated winning probability for the `posteam` given the current situation at the start of the given play.

<br>

## Question 8
In DataFrame, `NFL2023_stuffs`, remove observations for which values of `posteam` is missing.

*Answer*:

In [40]:
len(NFL2023_stuffs) == NFL2023_stuffs['posteam'].notna().sum()

False

In [41]:
q8 = (
    NFL2023_stuffs
    .dropna(subset = ['posteam'])
)

len(q8) == q8['posteam'].notna().sum()

True

<br>


## Question 9
- Calculate the mean value of `pass` for each `posteam` when all the following conditions hold:
  1. `wp` is greater than 20% and less than 75%;
  2. `down` is less than or equal to 2; and
  3. `half_seconds_remaining` is greater than 120.

*Answer*:

In [44]:
q9 = (
    NFL2023_stuffs
    .query('wp > .2 & wp < .75 & down <= 2 & half_seconds_remaining > 120')
    .groupby('posteam')[['pass']]
    .agg(
        pass_mean = ('pass', 'mean')
    ).
    sort_values('pass_mean', ascending = False)
)

show(q9)